## 1. 인증 설정
`.env`에서 NCP API 키를 불러와 블로그 검색 API용 헤더, 스크래핑용 User-Agent 헤더를 만듦.

In [1]:
import os
import re
import time

import dotenv
import pandas as pd
import requests
from bs4 import BeautifulSoup

dotenv.load_dotenv()

client_id = os.getenv("client_ID")
client_secret = os.getenv("client_secret")

BLOG_SEARCH_URL = "https://naverapihub.apigw.ntruss.com/search/v1/blog"
SEARCH_HEADERS = {
    "X-NCP-APIGW-API-KEY-ID": client_id,
    "X-NCP-APIGW-API-KEY": client_secret,
}
SCRAPE_HEADERS = {"User-Agent": "Mozilla/5.0"}

## 2. 블로그 검색 함수
검색 API를 `start`/`display`로 페이지네이션하며 제목·요약·블로거·작성일을 수집 (`description`은 요약본이라 전체 내용은 아님).

In [2]:
def clean_html(text):
    return re.sub(r"<.*?>", "", text) if isinstance(text, str) else text


def search_naver_blog(query, total=100, sort="date"):
    """블로그 검색 API를 start/display로 페이지네이션하며 최대 total건 수집 (title/description은 요약본)"""
    rows = []
    display = 100
    for start in range(1, total + 1, display):
        params = {
            "query": query,
            "display": min(display, total - start + 1),
            "start": start,
            "sort": sort,
            "format": "json",
        }
        resp = requests.get(BLOG_SEARCH_URL, headers=SEARCH_HEADERS, params=params)
        if resp.status_code != 200:
            print(f"검색 실패: {query} start={start} ({resp.status_code})")
            break

        items = resp.json().get("items", [])
        if not items:
            break

        for item in items:
            rows.append({
                "title": clean_html(item.get("title")),
                "link": item.get("link"),
                "description": clean_html(item.get("description")),
                "bloggername": item.get("bloggername"),
                "bloggerlink": item.get("bloggerlink"),
                "postdate": item.get("postdate"),
                "query": query,
            })

        time.sleep(0.2)  # API 호출 제한 방지

    return pd.DataFrame(rows)

## 3. 블로그 본문 스크래핑 함수
검색 결과 링크의 `PostView.naver` 페이지를 다시 요청해 본문 전체 텍스트를 추출 (장소/기간/시간 등 요약본에서 잘린 정보 확보용).

In [3]:
def parse_blog_ids(link):
    """블로그 글 링크에서 blogId, logNo 추출"""
    m = re.search(r"blog\.naver\.com/([^/?]+)/(\d+)", link)
    if not m:
        return None, None
    return m.group(1), m.group(2)


def scrape_blog_content(link, timeout=10):
    """블로그 글 본문 전체 텍스트 스크래핑 (검색 API의 description은 요약이라 장소/시간 등이 잘릴 수 있음)"""
    blog_id, log_no = parse_blog_ids(link)
    if not blog_id:
        return None

    post_url = f"https://blog.naver.com/PostView.naver?blogId={blog_id}&logNo={log_no}"
    try:
        resp = requests.get(post_url, headers=SCRAPE_HEADERS, timeout=timeout)
    except requests.RequestException:
        return None
    if resp.status_code != 200:
        return None

    soup = BeautifulSoup(resp.text, "lxml")
    main = soup.select_one("div.se-main-container") or soup.select_one("#postViewArea")
    if not main:
        return None

    return main.get_text("\n", strip=True)

## 4. 실행: 검색어별 동적 수집 + 누적 저장
`AREA`는 고정, `KEYWORDS` 리스트에 항목만 추가/변경하면 됨(지역+키워드 조합은 자동 생성). 이미 저장된 `link`는 스킵(재요청 안 함)하고, 새 결과만 기존 CSV와 병합 후 `link` 기준 중복 제거해서 저장.

In [ ]:
# 동적 처리: 지역 x 키워드 조합만 바꿔가며 재실행 -> 검색 + 본문 스크래핑까지 누적 저장
from nolda_common import MAPO_DONGS

CSV_PATH = "data/naver_blog_content.csv"
TOTAL_PER_QUERY = 100

# 관심 동네 전체를 훑고 싶으면 MAPO_DONGS 그대로, 특정 동만 보고 싶으면 부분 리스트로 교체, 다른 키워드 추가 시 리스트 추가
AREAS = MAPO_DONGS + ['마포구']
KEYWORDS = [
    "팝업스토어",
]
QUERIES = [f"{area} {kw}" for area in AREAS for kw in KEYWORDS]

existing_df = pd.read_csv(CSV_PATH) if os.path.exists(CSV_PATH) else pd.DataFrame(columns=["link"])
seen_links = set(existing_df["link"])

rows = []
for query in QUERIES:
    df_search = search_naver_blog(query, total=TOTAL_PER_QUERY)
    for _, item in df_search.iterrows():
        link = item["link"]
        if link in seen_links:
            continue  # 이미 수집한 글은 재요청하지 않음

        content = scrape_blog_content(link)
        rows.append({**item.to_dict(), "content": content})
        seen_links.add(link)
        time.sleep(0.3)  # 블로그 서버 부하 방지

new_df = pd.DataFrame(rows)
df_all = pd.concat([existing_df, new_df], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["link"]).reset_index(drop=True)
df_all.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

print(f"신규 스크래핑 {len(new_df)}건 / 총 {len(df_all)}건 저장 (본문 스크래핑 실패: {df_all['content'].isna().sum()}건)")
df_all.head()